# 0.4 随机性与模拟：量化交易的“排雷”演习

> **“如果再来一次，我的策略还会赚钱吗？”**
> 
> 回测只是“这辈子”发生过的事。但金融市场是充满随机性的，如果历史稍微偏差一点点，你的策略可能就会崩盘。
> 
> **这一节教你用概率和模拟，给你的策略做“压力测试”。**

## 学习目标
- 直观理解概率分布，特别是金融中的“黑天鹅”（厚尾）
- 掌握大数定律（LLN）如何保障回测的有效性
- 手写一个蒙特卡洛模拟（Monte Carlo），生成 100 种可能的未来

## 1. 概率分布：正态分布是真的吗？

大多数金融教科书假设收益率服从**正态分布（钟形曲线）**。这意味着极端事件（如 $-10\%$ 的单日跌幅）几百年才发生一次。

但在真实市场中，极端事件发生的频率远高于正态分布的预测。这就是所谓的**“厚尾”（Heavy-Tailed）**。

**核心直觉**：在金融市场，你必须时刻准备应对“概率极低但杀伤力极大”的黑天鹅事件。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import yfinance as yf

# 我们来看看 SPY 的真实收益率分布
data = yf.download('SPY', start='2010-01-01', end='2024-01-01', progress=False)['Close']
returns = data.pct_change().dropna()

plt.figure(figsize=(10, 6))
plt.hist(returns, bins=100, density=True, alpha=0.6, color='skyblue', label='真实收益率分布')

# 叠加一个完美的正态分布
mu, std = returns.mean(), returns.std()
x = np.linspace(returns.min(), returns.max(), 100)
plt.plot(x, stats.norm.pdf(x, mu, std), 'r-', lw=2, label='理论正态分布')

plt.title("真实分布 vs 理论正态分布：注意两侧的‘厚尾’")
plt.legend()
plt.show()

print(f"真实峰度 (Kurtosis): {returns.kurtosis():.2f}")
print("正态分布的峰度是 3。真实市场峰度远大于 3，意味着极端涨跌比理论上多得多。")

## 2. 大数定律（LLN）：回测的信心来源

**大数定律说明**：只要你回测的数据足够多（样本量大），你看到的“平均收益率”就会越来越接近真实的“概率期望”。

这就是为什么我们回测不能只看 10 天，而要看 5-10 年。样本越多，偶然性就越低。

In [ ]:
# 演示：抛硬币（模拟 50% 胜率策略）
np.random.seed(42)
n_trials = 1000
flips = np.random.choice([1, -1], size=n_trials)
running_mean = np.cumsum(flips) / np.arange(1, n_trials + 1)

plt.figure(figsize=(10, 4))
plt.plot(running_mean)
plt.axhline(0, color='red', linestyle='--')
plt.title("大数定律：随着成交次数增加，胜率趋于稳定")
plt.xlabel("交易次数")
plt.ylabel("平均收益")
plt.show()

## 3. 蒙特卡洛模拟：预测 100 种未来

**核心思路**：虽然我们不知道未来精确的价格，但我们可以通过“扔骰子”模拟出无数种符合当前特征（均值、波动率）的随机走势。  
如果你的策略在 100 种模拟的未来里，有 90 种都是赚钱的，那么这种策略的稳健性就很高。
如果它在 30 种走势里都亏得倾家荡产，那你就要小心了。

In [ ]:
# 简单的蒙特卡洛模拟
last_price = data.iloc[-1]
days = 252 # 预测未来一年
n_sims = 100 # 模拟 100 个版本

avg_daily_ret = returns.mean()
daily_vol = returns.std()

sim_returns = np.random.normal(avg_daily_ret, daily_vol, (days, n_sims))
sim_price_paths = last_price * (1 + sim_returns).cumprod(axis=0)

plt.figure(figsize=(12, 6))
plt.plot(sim_price_paths, color='gray', alpha=0.1)
plt.plot(sim_price_paths.mean(axis=1), color='red', linewidth=3, label='平均预期走势')
plt.title(f"SPY 未来 {days} 天的蒙特卡洛模拟（100 种可能）")
plt.legend()
plt.show()

print(f"模拟中最差的一种情况，价格跌到了: {sim_price_paths.min():.2f}")
print(f"模拟中最好的一种情况，价格涨到了: {sim_price_paths.max():.2f}")

## 🎯 小结

- **分布不是死板的公式**：金融中的“厚尾”决定了你必须防范大崩盘。
- **大数定律是底气**：回测样本越多，结论越可靠。
- **模拟是最好的演习**：在实盘投钱之前，先在模拟的一百个宇宙里跑一遍。

---
**下一节** → `../02_data/`（我们将学习如何获取更严谨的高质量历史数据）